In [53]:
import numpy as np
import sympy as sp
# Gates
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
P0 = np.array([[1, 0], [0, 0]], dtype=complex)   # |0><0|
P1 = np.array([[0, 0], [0, 1]], dtype=complex)   # |1><1|
def Ry(theta):
    c, s = np.cos(theta/2), np.sin(theta/2)
    return np.array([[c, -s], [s, c]], dtype=complex)



class QuantumCircuit:
    def __init__(self, n):
        self.n = n
        self.state = np.zeros(2**self.n, dtype=complex)
        self.state[0] = 1

    def apply(self, gate, k):
        op = np.array([[1]], dtype=complex)
        for i in range(self.n):
            if i == k:
                op = np.kron(op, gate)
            else:
                op = np.kron(op, I)
        self.state = op @ self.state
    
    def cnot(self, control, target):
        op0 = np.array([[1]], dtype=complex)
        op1 = np.array([[1]], dtype=complex)
        for i in range(self.n):
            if i == control:
                op0 = np.kron(op0, P0)
                op1 = np.kron(op1, P1)
            elif i == target:
                op0 = np.kron(op0, I)
                op1 = np.kron(op1, X)
            else:
                op0 = np.kron(op0, I)
                op1 = np.kron(op1, I)
        self.state = (op0 + op1) @ self.state
    
    def probabilities(self):
        return np.abs(self.state)**2


    def prob_one(self, k):
        probs = self.probabilities()
        total = 0
        for idx, p in enumerate(probs):
            bits = format(idx, f"0{self.n}b")
            if bits[k] == "1":
                total += p
        return total

        
    def cz(self, control, target):
        self.apply(H, target)
        self.cnot(control, target)
        self.apply(H, target)

    def measure(self, shots=1000):
        probs = self.probabilities()
        outcomes = np.random.choice(len(probs), size=shots, p=probs)
        counts = {}
        for idx in outcomes:
            bits = format(idx, f"0{self.n}b")     # e.g. 3 -> "11"
            counts[bits] = counts.get(bits, 0) + 1
        return counts


In [54]:
qc = QuantumCircuit(2)
qc.apply(H, 0)
print(qc.state)      # want [0.707, 0, 0.707, 0]

qc2 = QuantumCircuit(2)
qc2.apply(X, 1)
print(qc2.state)  

[0.70710678+0.j 0.        +0.j 0.70710678+0.j 0.        +0.j]
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]


In [55]:
bell = QuantumCircuit(2)
bell.apply(H, 0)
bell.cnot(0, 1)
print(bell.state)

[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


In [56]:
# Test
bell = QuantumCircuit(2)
bell.apply(H, 0)
bell.cnot(0, 1)
print(bell.probabilities())   # [0.5, 0, 0, 0.5]
print(bell.measure(1000)) 

[0.5 0.  0.  0.5]
{'11': 532, '00': 468}


In [57]:
theta = 1.2                          # pick any angle

qc = QuantumCircuit(3)
qc.apply(Ry(theta), 0)               # 1. make |psi> on q0

qc.apply(H, 1)                       # 2. Bell pair on q1, q2
qc.cnot(1, 2)

qc.cnot(0, 1)                        # 3. Alice's side
qc.apply(H, 0)

qc.cnot(1, 2)                        # 4. Bob's fixes
qc.cz(0, 2)

print("Expected P(1):", np.sin(theta/2)**2)
print("Bob's q2 P(1):", qc.prob_one(2))

Expected P(1): 0.31882112276166324
Bob's q2 P(1): 0.31882112276166286
